PCA/GRADIENT BOOSTING

Jack Silkaitis

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor

from sklearn.metrics import mean_squared_error
import plotly.express as px
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.decomposition import PCA

In [20]:
# Import data
#df = pd.read_csv('./CleanPreprocessedExoplanet.csv')
df = pd.read_csv('./CleanPreprocessedExoplanet2.csv')
df.head(3)
df.shape

(1788, 51)

In [21]:
df.columns

Index(['Unnamed: 0', 'release_month', 'sy_snum', 'sy_pnum', 'cb_flag',
       'ttv_flag', 'pl_nnotes', 'elon', 'elat', 'glon', 'glat', 'dec', 'ra',
       'st_nspec', 'pl_nespec', 'pl_ntranspec', 'pl_ndispec', 'release_year',
       'st_nphot', 'st_nrvc', 'pl_controv_flag', 'disc_year', 'sy_tmag',
       'sy_kmag', 'sy_jmag', 'sy_hmag', 'sy_vmag', 'sy_pm', 'sy_pmdec',
       'sy_pmra', 'pl_orbper', 'sy_gaiamag', 'st_rad', 'sy_bmag', 'sy_dist',
       'st_teff', 'sy_w2mag', 'sy_w1mag', 'sy_w3mag', 'sy_w4mag', 'sy_plx',
       'st_logg', 'st_mass',
       'soltype_Kepler Project Candidate (q1_q17_dr25_koi)',
       'soltype_Published Confirmed', 'pl_tsystemref_BJD',
       'pl_tsystemref_BJD-TDB', 'pl_tsystemref_BJD-UTC', 'pl_tsystemref_HJD',
       'pl_tsystemref_JD', 'log radius'],
      dtype='object')

In [22]:
# Split and prepare data
(df_train,df_test) = train_test_split(df,train_size=0.8,
                                      test_size=0.2,
                                      random_state=0)

In [23]:
X_train = df_train.drop(['log radius','Unnamed: 0'],axis=1)
y_train = df_train['log radius']
X_test = df_test.drop(['log radius','Unnamed: 0'],axis=1)
y_test = df_test['log radius']

In [24]:
stnd = StandardScaler().set_output(transform='pandas')

X_number = X_train.select_dtypes(include='number')
X_categorical = X_train.select_dtypes(exclude='number')
X_number = stnd.fit_transform(X_number)
X_train = pd.concat([X_number, X_categorical], axis=1)

X_number = X_test.select_dtypes(include='number')
X_categorical = X_test.select_dtypes(exclude='number')
X_number = stnd.transform(X_number)
X_test = pd.concat([X_number, X_categorical], axis=1)

In [25]:
# Baseline before applying PCA, arbitrary max_depth
tree = DecisionTreeRegressor(max_depth=2)
tree.fit(X_train,y_train)
tree.score(X_test, y_test)

0.13648169148643174

In [26]:
# Apply PCA
n_comp = X_train.shape[1]
cols = ['PC-'+str(i+1) for i in range(n_comp)]
cols

pca = PCA(n_components=n_comp)
pca.fit(X_train)
PVE = pca.explained_variance_ratio_
PVE = pd.Series(PVE,index=cols)
print(PVE.round(2))

PC-1     0.33
PC-2     0.12
PC-3     0.06
PC-4     0.04
PC-5     0.04
PC-6     0.03
PC-7     0.03
PC-8     0.03
PC-9     0.03
PC-10    0.03
PC-11    0.02
PC-12    0.02
PC-13    0.02
PC-14    0.02
PC-15    0.02
PC-16    0.02
PC-17    0.02
PC-18    0.02
PC-19    0.01
PC-20    0.01
PC-21    0.01
PC-22    0.01
PC-23    0.01
PC-24    0.01
PC-25    0.01
PC-26    0.01
PC-27    0.00
PC-28    0.00
PC-29    0.00
PC-30    0.00
PC-31    0.00
PC-32    0.00
PC-33    0.00
PC-34    0.00
PC-35    0.00
PC-36    0.00
PC-37    0.00
PC-38    0.00
PC-39    0.00
PC-40    0.00
PC-41    0.00
PC-42    0.00
PC-43    0.00
PC-44    0.00
PC-45    0.00
PC-46    0.00
PC-47    0.00
PC-48    0.00
PC-49    0.00
dtype: float64


In [27]:
PVE_sort = PVE.sort_values(ascending=False)
PVE_sort[0:4].sum()

np.float64(0.5594821252813291)

In [28]:
for i in range(n_comp):
    sum = PVE_sort[0:i].sum()
    if sum > 0.9:
        print(i)
        break

19


In [29]:
n_comp = 19
cols = ['PC-'+str(i+1) for i in range(n_comp)]
pca = PCA(n_components=n_comp)
pca.fit(X_train)
X_train_pca = pca.transform(X_train)
X_test_pca = pca.transform(X_test)

In [30]:
B = np.arange(1,301,100)
C = np.arange(1,10,3)
D = np.arange(0.01,1,0.33)
grid = {'n_estimators':B, 'max_depth':C, 'learning_rate':D}

gbt = GradientBoostingRegressor()
gbtCV = GridSearchCV(gbt,param_grid=grid,return_train_score=True,n_jobs=-1, verbose=2)
gbtCV.fit(X_train_pca,y_train)

print('best params =',gbtCV.best_params_, '  valid acc =',gbtCV.best_score_)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
best params = {'learning_rate': np.float64(0.01), 'max_depth': np.int64(7), 'n_estimators': np.int64(201)}   valid acc = 0.3107459064624397


In [32]:
B = np.arange(150,300,50)
C = np.arange(6,9,1)
D = np.arange(.01, .33, .07)
grid = {'n_estimators':B, 'max_depth':C, 'learning_rate':D}

gbt = GradientBoostingRegressor()
gbtCV = GridSearchCV(gbt,param_grid=grid,return_train_score=True,n_jobs=-1, verbose=2)
gbtCV.fit(X_train_pca,y_train)

print('best params =',gbtCV.best_params_, '  valid acc =',gbtCV.best_score_)

Fitting 5 folds for each of 45 candidates, totalling 225 fits
best params = {'learning_rate': np.float64(0.01), 'max_depth': np.int64(6), 'n_estimators': np.int64(250)}   valid acc = 0.31715780753460965


In [33]:
B = np.arange(200,276,25)
C = np.arange(.01, .07,.01)
grid = {'n_estimators':B, 'learning_rate':C}

gbt = GradientBoostingRegressor(max_depth=6)
gbtCV = GridSearchCV(gbt,param_grid=grid,return_train_score=True,n_jobs=-1, verbose=2)
gbtCV.fit(X_train_pca,y_train)

print('best params =',gbtCV.best_params_, '  valid acc =',gbtCV.best_score_)

Fitting 5 folds for each of 24 candidates, totalling 120 fits
best params = {'learning_rate': np.float64(0.02), 'n_estimators': np.int64(200)}   valid acc = 0.32073421509844274


In [34]:
bestGBT = GradientBoostingRegressor(max_depth=6, n_estimators=200, learning_rate=.02)

In [35]:
print('Test R2',gbtCV.score(X_test_pca,y_test))
print('Train R2',gbtCV.score(X_train_pca,y_train))

Test R2 0.3640241768150817
Train R2 0.8370203787554664


In [37]:
y_pred = gbtCV.predict(X_train_pca)
mean_squared_error(y_pred, y_train)

0.012589550652813721

In [38]:
#bestGBT.fit(X_train_pca, y_train)
y_pred = gbtCV.predict(X_test_pca)
mean_squared_error(y_pred, y_test)

0.045547804395361165

In [39]:
mean_squared_error(10**y_pred, 10**y_test)

4.933337921816578

In [40]:
np.mean(10**(y_pred) / 10**y_test)

np.float64(1.102112334014617)

In [41]:
np.mean(np.abs(10**(y_pred) - 10**y_test) / 10**y_test)

np.float64(0.38017078241856317)